# Stage 3 — ESCO Occupation Classification (Batch API)
## Notebook 3.5 of 7 — Create Batch Input Files for Missing Records

**Role in the pipeline:**
A second classification pass for vacancies that were not successfully classified in the first pass (notebooks 3.1–3.4). This notebook creates new JSONL batch input files for those missing records, saving them to the `missing/` subfolder under `STAGE3_INPUT_PATH`.

Key differences from notebook 3.1:
- Uses `missing_path` pickles (not `extract_path`) as input — these contain only unclassified vacancies.
- Skips rows where `missing_input_batch_status == 'empty'` (no missing records for that file).
- Uses a more capable model (`gpt-4o`) for the retry attempt.

**Position in Stage 3 sequence:**
1. 3.1 — Create batch input files
2. 3.2 — Submit batch jobs and monitor status
3. 3.3 — Extract classification results
4. 3.4 — Split missing and complete classifications
5. ▶ **3.5 — Create batch input files for missing records** ← *you are here*
6. 3.6 — Submit batch jobs for missing records *(run after Batch API completes)*
7. 3.7 — Extract results for missing records

**Prerequisites:**
- Notebook 3.4 completed — missing pickles exist in `STAGE3_RESULT_PATH/missing/`

## 3.1.2. Load packages and configuration

Loads standard libraries and the shared `general` config module.

In [ ]:
import pandas as pd
%load_ext autoreload
%autoreload 2
import sys
import os
sys.path.append("../code")
import general as g
g.clean_memory()

Imports Stage 2/3 processing modules, the OpenAI client, and `random` for model selection.

In [ ]:
import stage2 as st2
import stage3 as st3
from openai import OpenAI
import pandas as pd

## 3.1.3. Load processing log

Reads the current tracker. The commented-out lines show how to initialise `missing_input_batch_status` and `missing_input_batch_path` columns for the first run of this notebook.

In [ ]:
process_df = pd.read_pickle(g.Config.STAGE3_PROCESS_PATH)
process_df

## 3.1.4. Create JSONL input files for missing records

Initialises the OpenAI client.

In [ ]:
client = OpenAI(api_key=g.Config.OPENAI_API_KEY)

Main loop for missing record batch file creation:

- Rows with `missing_input_batch_status == 'created'` or `'empty'` are skipped.
- Rows where `missing_count == 0` are marked `empty` with a placeholder path `'-'` — no batch file is needed.
- For rows with missing records: reads the missing pickle, builds the classification payload, and writes a JSONL file to `STAGE3_INPUT_PATH/missing/`.

Note: the retry uses `gpt-4o` (more capable than `gpt-4.1-mini` used in notebook 3.1) to improve coverage for hard-to-classify vacancies.

In [ ]:
process_df = pd.read_pickle(g.Config.STAGE3_PROCESS_PATH)

for _,row in process_df.iterrows():
   # print("Processing", row["input_file"])
    if row["missing_input_batch_status"] == "created" or row["missing_input_batch_status"] == "empty":
        continue
    elif row["missing_count"] == 0:
        process_df.loc[_, "missing_input_batch_path"] = "-"
        process_df.loc[_, "missing_input_batch_status"] = "empty"
    else:
        file_path = process_df.loc[_, "missing_path"]
        skills_df = st3.get_extracted_skills_df(file_path)
        class_schema = st3.read_classification_schema(g.Config.STAGE3_CLASSIFY_SCHEME)
        class_prompt = st3.read_classification_prompt(g.Config.STAGE3_CLASSIFY_PROMPT)

        batch_file_path = os.path.join(g.Config.STAGE3_INPUT_PATH, "missing", process_df.loc[_, "input_file"] + ".jsonl")
        batch_file_path = st3.write_batch_file(batch_file_path, skills_df, class_schema, class_prompt, "gpt-4o-mini")

        process_df.loc[_, "missing_input_batch_path"] = batch_file_path
        process_df.loc[_, "missing_input_batch_status"] = "created"

process_df.to_pickle(g.Config.STAGE3_PROCESS_PATH)

Displays the updated tracker to confirm that `missing_input_batch_status` has been populated for all rows.

In [ ]:
process_df

---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.